# EvoVariant-TR foundation-model fine-tuning attempt

This is a read-only evidence notebook. It demonstrates the implementation and the real encoder-update proof without launching a long run. The proof is feasibility evidence, not completed fine-tuning.

In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd()
if not (ROOT / "research/adaptation_attempt").exists() and Path("/content/EvoVariant/research/adaptation_attempt").exists():
    ROOT = Path("/content/EvoVariant")
APP = ROOT / "research/adaptation_attempt"
proof = json.loads((APP / "artifacts/caduceus_partial_small_finetune_smoke.json").read_text())
manifest = json.loads((APP / "EVIDENCE_MANIFEST.json").read_text())
print(json.dumps({"proof": proof, "boundary": manifest["data_boundary"]}, indent=2))

## Model and trainability configuration

The following cell prints the committed source that freezes the earlier Caduceus blocks and unfreezes the declared final blocks. Source visibility is not counted as a run.

In [ ]:
models_source = (APP / "source_snapshot/models.py").read_text()
start = models_source.find("def configure_trainable")
end = models_source.find("def trainable_parameter_manifest", start)
print(models_source[start:end])
print("model:", proof["model_id"])
print("revision:", proof["model_revision"])
print("trainable blocks:", proof["trainable_blocks"])
print("frozen blocks:", proof["frozen_blocks"])

## Actual gradient/update path

The implementation contains a real loss, backward(), gradient measurement, and optimizer.step(). This cell shows the relevant source and the recorded nonzero/frozen deltas.

In [ ]:
training_source = (APP / "source_snapshot/training.py").read_text()
for marker in ("loss.backward()", "optimizer.step()", "train_loss"):
    position = training_source.find(marker)
    print("\n---", marker, "---")
    print(training_source[max(0, position - 350): position + 450] if position >= 0 else "marker not found")
print("encoder delta norm:", proof["encoder_delta_norm"])
print("frozen control delta norm:", proof["frozen_control_delta_norm"])
assert proof["encoder_delta_norm"] > 0
assert proof["frozen_control_delta_norm"] == 0

## HPO queue and measured state

In [ ]:
hpo_source = (APP / "source_snapshot/run_caduceus_hpo.py").read_text()
for marker in ("frozen_head_only", "partial_small", "partial_large", "full_if_feasible", "StratifiedGroupKFold"):
    print(marker, "present:", marker in hpo_source)
print("completed fine-tuning experiment:", proof["completed_fine_tuning_experiment"])
print("selection closed:", proof["selection_closed"])
print("801 holdout evaluated:", proof["holdout_evaluated"])
print("Allowed conclusion: encoder-update feasibility proof only")

## Final status

Frozen-head TRAIN-only training completed. Partial-small, partial-large, and full foundation-model fine-tuning did not complete; adaptation HPO remained open and resource-limited. No 801-row adaptation metric is displayed or implied.